# Taller Week 6: Encoder-Decoder con Atención — CNN/DailyMail

**Tarea:** Resumen automático de noticias usando la arquitectura Encoder-Decoder + Atención + GloVe embeddings.

**Dataset:** `cnn_dailymail` (Hugging Face) — campo `article` como entrada, `highlights` como salida.

## 1. Imports y configuración

In [1]:
# ── EXPERIMENT CONFIG ────────────────────────────────────────────────────
EXPERIMENT_NAME = "v1_baseline"  # <-- change this each run
# ─────────────────────────────────────────────────────────────────────────

import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Embedding, Attention, Concatenate, TimeDistributed
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset

print("TensorFlow:", tf.__version__)


/Users/dpletzke/dev/ml-nlp-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow: 2.21.0


In [ ]:
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive')
    _base      = '/content/drive/MyDrive/ml-nlp-project'
    GLOVE_PATH = f'{_base}/glove.6B.300d.txt'
except ImportError:
    # Running locally
    _base      = '.'
    GLOVE_PATH = './data/glove/glove.6B.300d.txt'

RUN_ID  = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = os.path.join(_base, 'runs', f'{EXPERIMENT_NAME}_{RUN_ID}')
os.makedirs(RUN_DIR, exist_ok=True)
print(f'RUN_DIR: {RUN_DIR}')


## 2. Consulta sobre `cnn_dailymail`

`cnn_dailymail` es un dataset de Hugging Face construido con artículos de CNN y Daily Mail. Contiene tres columnas:
- `article`: texto completo del artículo periodístico.
- `highlights`: puntos principales / resumen humano del artículo.
- `id`: identificador único del ejemplo.

El problema que resolvemos es **resumen abstractivo automático**: dado un documento largo, generar una versión corta que conserve las ideas principales. Aplicaciones posibles:
- Resumen de noticias para lectores con poco tiempo.
- Procesamiento de reportes corporativos extensos.
- Asistentes de lectura para documentos académicos o legales.

## 3. Carga del dataset

In [2]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data      = dataset["train"]
validation_data = dataset["validation"]
test_data       = dataset["test"]

print(f"Train:      {len(train_data):,} ejemplos")
print(f"Validation: {len(validation_data):,} ejemplos")
print(f"Test:       {len(test_data):,} ejemplos")

Train:      287,113 ejemplos
Validation: 13,368 ejemplos
Test:       11,490 ejemplos


In [3]:
# Muestra de un ejemplo
example = train_data[0]
print("=== ARTICLE (primeros 500 chars) ===")
print(example["article"][:500])
print("\n=== HIGHLIGHTS ===")
print(example["highlights"])

=== ARTICLE (primeros 500 chars) ===
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

=== HIGHLIGHTS ===
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [4]:
# Muestra reducida para entrenamiento local
N_TRAIN = 30_000
N_VAL   = 1_000
N_TEST  = 1_000

train_sample = train_data.select(range(N_TRAIN))
val_sample   = validation_data.select(range(N_VAL))
test_sample  = test_data.select(range(N_TEST))

print(f"Usando {N_TRAIN} ejemplos de entrenamiento")

Usando 10000 ejemplos de entrenamiento


## 4. Preprocesamiento

In [5]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9?.!,\'\" ]+', ' ', text)
    return text.strip()

articles  = [clean_text(ex["article"])     for ex in train_sample]
summaries = ["<start> " + clean_text(ex["highlights"]) + " <end>" for ex in train_sample]

val_articles  = [clean_text(ex["article"])     for ex in val_sample]
val_summaries = ["<start> " + clean_text(ex["highlights"]) + " <end>" for ex in val_sample]

print("Ejemplo limpio:")
print("  Article:",  articles[0][:200])
print("  Summary:",  summaries[0][:200])

Ejemplo limpio:
  Article: london, england  reuters    harry potter star daniel radcliffe gains access to a reported  20 million  41.1 million  fortune as he turns 18 on monday, but he insists the money won't cast a spell on hi
  Summary: <start> harry potter star daniel radcliffe gets  20m fortune as he turns 18 monday . young actor says he has no plans to fritter his cash away . radcliffe's earnings from first five potter films have 


In [6]:
# Longitudes máximas (cap para no exceder memoria)
MAX_ARTICLE_LEN = 400
MAX_SUMMARY_LEN = 50
VOCAB_SIZE_ARTICLE = 30_000

# Tokenización de artículos
art_tokenizer = Tokenizer(num_words=VOCAB_SIZE_ARTICLE, filters='', oov_token='<unk>')
art_tokenizer.fit_on_texts(articles)

# Tokenización de resúmenes — sin cap ni oov_token para eliminar <unk> de los targets
sum_tokenizer = Tokenizer(filters='')
sum_tokenizer.fit_on_texts(summaries)
VOCAB_SIZE_SUMMARY = len(sum_tokenizer.word_index) + 1

print(f"Vocab artículos: {len(art_tokenizer.word_index):,}")
print(f"Vocab resúmenes: {len(sum_tokenizer.word_index):,}")
print(f"VOCAB_SIZE_SUMMARY: {VOCAB_SIZE_SUMMARY:,}")
print("<unk> in summary vocab:", '<unk>' in sum_tokenizer.word_index)


In [7]:
# Secuencias y padding
enc_input_train = pad_sequences(
    art_tokenizer.texts_to_sequences(articles),
    maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post'
)

sum_seqs_train = pad_sequences(
    sum_tokenizer.texts_to_sequences(summaries),
    maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post'
)

# Decoder input = resumen con <start>; decoder target = resumen desplazado hasta <end>
dec_input_train  = sum_seqs_train[:, :-1]   # drop last token
dec_target_train = sum_seqs_train[:, 1:]    # drop first token (<start>)
dec_target_train = dec_target_train[..., np.newaxis]  # sparse_categorical needs shape (..., 1)

print("Encoder input shape:", enc_input_train.shape)
print("Decoder input shape:", dec_input_train.shape)
print("Decoder target shape:", dec_target_train.shape)

Encoder input shape: (10000, 200)
Decoder input shape: (10000, 49)
Decoder target shape: (10000, 49, 1)


## 5. Embeddings preentrenados (GloVe 300d)

In [ ]:
import zipfile, urllib.request

if not os.path.exists(GLOVE_PATH):
    print('GloVe not found, downloading (~820 MB)...')
    _glove_dir = os.path.dirname(GLOVE_PATH)
    os.makedirs(_glove_dir, exist_ok=True)
    _zip_path = os.path.join(_glove_dir, 'glove.6B.zip')
    urllib.request.urlretrieve('https://nlp.stanford.edu/data/glove.6B.zip', _zip_path)
    with zipfile.ZipFile(_zip_path, 'r') as zf:
        zf.extract('glove.6B.300d.txt', _glove_dir)
    os.remove(_zip_path)
    print(f'GloVe saved to {GLOVE_PATH}')
else:
    print(f'GloVe found: {GLOVE_PATH}')


In [8]:
def load_glove(path):
    embeddings = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            vec  = np.array(parts[1:], dtype='float32')
            embeddings[word] = vec
    print(f"GloVe cargado: {len(embeddings):,} vectores")
    return embeddings

glove = load_glove(GLOVE_PATH)

GloVe cargado: 400,000 vectores


In [9]:
EMBED_DIM = 300

def build_embedding_matrix(tokenizer, glove, vocab_size, dim=300):
    matrix = np.zeros((vocab_size, dim))
    for word, i in tokenizer.word_index.items():
        if i < vocab_size and word in glove:
            matrix[i] = glove[word]
    return matrix

art_emb_matrix = build_embedding_matrix(art_tokenizer, glove, VOCAB_SIZE_ARTICLE)
sum_emb_matrix = build_embedding_matrix(sum_tokenizer, glove, VOCAB_SIZE_SUMMARY)

print("Article embedding matrix:", art_emb_matrix.shape)
print("Summary embedding matrix:", sum_emb_matrix.shape)

Article embedding matrix: (30000, 300)
Summary embedding matrix: (15000, 300)


## 6. Modelo Encoder-Decoder + Atención

In [10]:
LATENT_DIM     = 300
DEC_SEQ_LEN    = MAX_SUMMARY_LEN - 1  # matches dec_input shape

# --- ENCODER ---
encoder_inputs = Input(shape=(MAX_ARTICLE_LEN,), name="Enc_Input")
enc_emb = Embedding(
    VOCAB_SIZE_ARTICLE, EMBED_DIM,
    weights=[art_emb_matrix], trainable=False, name="Enc_Embedding"
)(encoder_inputs)
encoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="Enc_LSTM")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = Input(shape=(DEC_SEQ_LEN,), name="Dec_Input")
dec_emb_layer  = Embedding(
    VOCAB_SIZE_SUMMARY, EMBED_DIM,
    weights=[sum_emb_matrix], trainable=False, name="Dec_Embedding"
)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="Dec_LSTM")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# --- ATENCIÓN (estilo Luong) ---
attn_layer = Attention(name="Attention_Layer")
attn_out   = attn_layer([decoder_outputs, encoder_outputs])

decoder_concat = Concatenate(axis=-1, name="Concat_Layer")([decoder_outputs, attn_out])

decoder_dense  = TimeDistributed(
    Dense(VOCAB_SIZE_SUMMARY, activation='softmax'), name="Dec_Dense"
)
decoder_outputs_final = decoder_dense(decoder_concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs_final)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Enc_Input           │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dec_Input           │ (None, 49)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Enc_Embedding       │ (None, 200, 300)  │  9,000,000 │ Enc_Input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dec_Embedding       │ (None, 49, 300)   │  4,500,000 │ Dec_Input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Enc_LSTM (LSTM)     │ [(None, 200,      │    570,368 │ Enc_Embedding[0]… │
│                     │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dec_LSTM (LSTM)     │ [(None, 49, 256), │    570,368 │ Dec_Embedding[0]… │
│                     │ (None, 256),      │            │ Enc_LSTM[0][1],   │
│                     │ (None, 256)]      │            │ Enc_LSTM[0][2]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Attention_Layer     │ (None, 49, 256)   │          0 │ Dec_LSTM[0][0],   │
│ (Attention)         │                   │            │ Enc_LSTM[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Concat_Layer        │ (None, 49, 512)   │          0 │ Dec_LSTM[0][0],   │
│ (Concatenate)       │                   │            │ Attention_Layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dec_Dense           │ (None, 49, 15000) │  7,695,000 │ Concat_Layer[0][… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 22,335,736 (85.20 MB)

 Trainable params: 8,835,736 (33.71 MB)

 Non-trainable params: 13,500,000 (51.50 MB)

## 7. Entrenamiento

In [ ]:
enc_input_val = pad_sequences(
    art_tokenizer.texts_to_sequences(val_articles),
    maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post'
)
sum_seqs_val = pad_sequences(
    sum_tokenizer.texts_to_sequences(val_summaries),
    maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post'
)
dec_input_val  = sum_seqs_val[:, :-1]
dec_target_val = sum_seqs_val[:, 1:][..., np.newaxis]

history = model.fit(
    [enc_input_train, dec_input_train],
    dec_target_train,
    batch_size=64,
    epochs=10,
    validation_data=([enc_input_val, dec_input_val], dec_target_val)
)

model.save(os.path.join(RUN_DIR, 'summarizer.h5'))
with open(os.path.join(RUN_DIR, 'tokenizer_art.pkl'), 'wb') as f:
    pickle.dump(art_tokenizer, f)
with open(os.path.join(RUN_DIR, 'tokenizer_sum.pkl'), 'wb') as f:
    pickle.dump(sum_tokenizer, f)
with open(os.path.join(RUN_DIR, 'history.json'), 'w') as f:
    json.dump(history.history, f, indent=2)

print(f'Modelo y tokenizadores guardados en: {RUN_DIR}')


Epoch 1/10


## 8. Inferencia autoregresiva

In [ ]:
def generate_summary(article_text, max_len=MAX_SUMMARY_LEN - 1):
    cleaned = clean_text(article_text)
    seq = art_tokenizer.texts_to_sequences([cleaned])
    enc_in = pad_sequences(seq, maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')

    start_token = sum_tokenizer.word_index.get('<start>', 1)
    end_token   = sum_tokenizer.word_index.get('<end>',   2)

    decoded = [start_token]

    for _ in range(max_len):
        dec_in = pad_sequences([decoded], maxlen=max_len, padding='post')
        preds  = model.predict([enc_in, dec_in], verbose=0)  # (1, seq, vocab)
        unk_index = sum_tokenizer.word_index.get('<unk>')
        if unk_index and unk_index < preds.shape[-1]:
            preds[0, len(decoded) - 1, unk_index] = 0.0
        next_idx = int(np.argmax(preds[0, len(decoded) - 1, :]))
        if next_idx == end_token or next_idx == 0:
            break
        decoded.append(next_idx)

    words = [sum_tokenizer.index_word.get(i, '') for i in decoded[1:]]
    return ' '.join(w for w in words if w)


examples_path = os.path.join(RUN_DIR, 'examples.txt')
with open(examples_path, 'w') as f:
    for i in range(5):
        ex        = test_sample[i]
        generated = generate_summary(ex['article'])

        f.write(f'\n=== Ejemplo {i+1} ===\n')
        f.write('ARTICLE:\n' + ex['article'][:500] + '\n\n')
        f.write('RESUMEN REAL:\n' + ex['highlights'] + '\n\n')
        f.write('RESUMEN GENERADO:\n' + generated + '\n')

        print(f'\n=== Ejemplo {i+1} ===')
        print('ARTICLE (inicio):', ex['article'][:300])
        print('RESUMEN REAL:    ', ex['highlights'])
        print('RESUMEN GENERADO:', generated)

print(f'\nEjemplos guardados en: {examples_path}')


## 9. Evaluación con ROUGE (opcional)

In [ ]:
!pip install -q evaluate rouge_score

import evaluate

rouge = evaluate.load('rouge')

predictions = [generate_summary(test_sample[i]['article']) for i in range(50)]
references  = [test_sample[i]['highlights'] for i in range(50)]

results = rouge.compute(predictions=predictions, references=references)
for k, v in results.items():
    print(f"{k}: {v:.4f}")

## 10. Conclusión

Se realizaron tres experimentos para evaluar el comportamiento del modelo encoder-decoder con atención sobre CNN/DailyMail.

### Calidad observada de los resúmenes generados

Los resultados variaron significativamente entre versiones:

- **v1 (baseline, 10k ejemplos):** El modelo colapsó casi por completo, generando secuencias de tokens `<unk>` en prácticamente todos los casos. La pérdida de validación seguía descendiendo al final de la época 10 (val_loss: 3.996), lo que indica que el modelo necesitaba más entrenamiento y datos.

- **v2 (baseline, 30k ejemplos):** Con más datos de entrenamiento, el modelo mejoró: los resúmenes contienen palabras reales mezcladas con `<unk>`, y la estructura gramatical comienza a emerger. Sin embargo, el contenido generado no refleja el tema del artículo fuente. La pérdida de validación comenzó a estancarse alrededor de la época 7 (val_loss ≈ 3.63), señal de que el modelo agotó su capacidad de generalizar con la configuración actual.

- **v3 (sin `<unk>` en vocab de resúmenes, 30k ejemplos):** Al eliminar el token `<unk>` del vocabulario objetivo, los resúmenes generados contienen exclusivamente palabras reales y muestran coherencia gramatical superficial. No obstante, el contenido es completamente alucinado: el modelo produce texto plausible en inglés pero sin relación semántica con el artículo de entrada (p. ej., el artículo sobre la Corte Penal Internacional genera texto sobre leyes de Nueva York). Esto evidencia que el modelo aprende distribuciones de lenguaje general pero no logra alinear correctamente la atención con el contenido del encoder.

La evaluación cuantitativa con ROUGE sobre 50 ejemplos del conjunto de prueba (v3) confirma la baja calidad:

| Métrica   | Valor  |
|-----------|--------|
| ROUGE-1   | 0.1633 |
| ROUGE-2   | 0.0231 |
| ROUGE-L   | 0.1338 |
| ROUGE-Lsum| 0.1462 |

El ROUGE-2 de 0.023 es especialmente revelador: el modelo casi no genera bigramas que coincidan con los resúmenes de referencia. A modo de comparación, modelos de estado del arte como BART fine-tuneado en CNN/DailyMail alcanzan ROUGE-2 ≈ 0.21, es decir, casi 10 veces más. En ninguno de los tres experimentos se obtuvieron resúmenes con calidad aceptable para uso práctico.

### Limitaciones del modelo

- **Vocabulario restringido en encoder:** El artículo se tokeniza con un vocabulario de 30,000 palabras; cualquier término fuera de ese rango se mapea a `<unk>`, perdiendo información de nombres propios, términos técnicos y entidades clave para el resumen.
- **Truncamiento de artículos:** Los artículos se recortan a 400 tokens, pero el artículo promedio de CNN/DailyMail supera los 700 tokens, por lo que se descarta más del 40% del contenido en muchos casos.
- **Pocas épocas:** 10 épocas no son suficientes para que un modelo seq2seq con atención converja sobre un corpus de noticias de esta complejidad.
- **Embeddings congelados:** Los vectores GloVe no se actualizan durante el entrenamiento, lo que limita la adaptación del modelo al dominio periodístico.
- **Decodificación greedy:** La selección del token más probable en cada paso favorece frases genéricas y repetitivas, y no permite explorar hipótesis alternativas.
- **Subconjunto pequeño:** Se usaron 30,000 de los 287,113 ejemplos disponibles (~10%), lo que restringe la capacidad del modelo para aprender patrones de resumen.

### Posibles mejoras

- **Más datos y épocas:** Usar el dataset completo (287k ejemplos) con 20-30 épocas.
- **Embeddings entrenables:** Descongelar los pesos de GloVe después de las primeras épocas permitiría al modelo ajustar las representaciones al dominio.
- **Beam search:** Reemplazar la decodificación greedy por beam search mejoraría la fluidez y relevancia de los resúmenes generados.
- **Arquitecturas BiLSTM y Transformer:** Un Encoder BiLSTM puede mejorar sobre un LSTM unidireccional porque lee el articulo en ambas direcciones y captura mas contexto local y global. Sin embargo, los modelos Transformer con atencion multi-cabeza, como BART o T5 fine-tuneados sobre CNN/DailyMail, suelen superar ampliamente a los LSTM/BiLSTM en resumen automatico porque modelan mejor dependencias de largo alcance y procesan la secuencia con menos degradacion.
- **Mayor longitud de entrada:** Aumentar `MAX_ARTICLE_LEN` o usar codificación jerárquica (párrafo → documento) para preservar más contenido del artículo original.

## 11. Ejemplos generados

### Ejemplo 1

**ARTICLE:**

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, includin

**RESUMEN REAL:**

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

**RESUMEN GENERADO:**

new the united states is a new york law to investigate the case . the case is a new human rights group says . the united states is a court court . the united states is a new version of the united arab emirates .

### Ejemplo 2

**ARTICLE:**

(CNN)Never mind cats having nine lives. A stray pooch in Washington State has used up at least three of her own after being hit by a car, apparently whacked on the head with a hammer in a misguided mercy killing and then buried in a field -- only to survive. That's according to Washington State University, where the dog -- a friendly white-and-black bully breed mix now named Theia -- has been receiving care at the Veterinary Teaching Hospital. Four days after her apparent death, the dog managed 

**RESUMEN REAL:**

Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field .
"She's a true miracle dog and she deserves a good life," says Sara Mellado, who is looking for a home for Theia .

**RESUMEN GENERADO:**

new the father of the year of the year old daughter was found dead in a florida hospital, florida . he was found in a car parked rv in a small town of the last year . the boy was found in a car that was found in a

### Ejemplo 3

**ARTICLE:**

(CNN)If you've been following the news lately, there are certain things you doubtless know about Mohammad Javad Zarif. He is, of course, the Iranian foreign minister. He has been U.S. Secretary of State John Kerry's opposite number in securing a breakthrough in nuclear discussions that could lead to an end to sanctions against Iran -- if the details can be worked out in the coming weeks. And he received a hero's welcome as he arrived in Iran on a sunny Friday morning. "Long live Zarif," crowds c

**RESUMEN REAL:**

Mohammad Javad Zarif has spent more time with John Kerry than any other foreign minister .
He once participated in a takeover of the Iranian Consulate in San Francisco .
The Iranian foreign minister tweets in English .

**RESUMEN GENERADO:**

the iranian president mahmoud ahmadinejad says he was not to be a new york . the iranian president mahmoud ahmadinejad says he was not to be a mistake . he says the irrational of the country is not to be a mistake .

### Ejemplo 4

**ARTICLE:**

(CNN)Five Americans who were monitored for three weeks at an Omaha, Nebraska, hospital after being exposed to Ebola in West Africa have been released, a Nebraska Medicine spokesman said in an email Wednesday. One of the five had a heart-related issue on Saturday and has been discharged but hasn't left the area, Taylor Wilson wrote. The others have already gone home. They were exposed to Ebola in Sierra Leone in March, but none developed the deadly virus. They are clinicians for Partners in Healt

**RESUMEN REAL:**

17 Americans were exposed to the Ebola virus while in Sierra Leone in March .
Another person was diagnosed with the disease and taken to hospital in Maryland .
National Institutes of Health says the patient is in fair condition after weeks of treatment .

**RESUMEN GENERADO:**

the outbreak occurred in the first time of the h1n1 virus . the outbreak is a contagious disease . the outbreak is a contagious of the virus .

### Ejemplo 5

**ARTICLE:**

(CNN)A Duke student has admitted to hanging a noose made of rope from a tree near a student union, university officials said Thursday. The prestigious private school didn't identify the student, citing federal privacy laws. In a news release, it said the student was no longer on campus and will face student conduct review. The student was identified during an investigation by campus police and the office of student affairs and admitted to placing the noose on the tree early Wednesday, the univer

**RESUMEN REAL:**

Student is no longer on Duke University campus and will face disciplinary review .
School officials identified student during investigation and the person admitted to hanging the noose, Duke says .
The noose, made of rope, was discovered on campus about 2 a.m.

**RESUMEN GENERADO:**

university student student was found dead in a parking lot of school . students were found in the school students . students were found in the university of the university student .